# Document loaders and text splitters

Assistant version 4: Load policies and handbook with source/page metadata; compare three chunking strategies.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --extra rag --group authoring`, then `uv run --extra rag --group authoring jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## Document.page_content and metadata; source provenance

Predict the output before running. The function definitions below match `examples/01_documents.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Keep text and source provenance together in a Document.

Run: uv run --extra rag python modules/06_loaders_splitters/examples/01_documents.py --offline
Expected: Return-policy text, a stable ID, and a relative source path.
Cost: No model calls or API fees.
Requires: rag
"""

from common.errors import run_cli
from common.knowledge import LocalLoader
from common.lesson import lesson_settings
from common.pretty import print_text


def main() -> None:
    lesson_settings()
    doc = LocalLoader("policies/returns.md").load()[0]
    # KEY IDEA: page_content is evidence; metadata tells us where it came from.
    print_text(doc.page_content[:180])
    print_text(str({"id": doc.id, **doc.metadata}))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Current BaseLoader PDF adapter with pypdf; lazy_load, one-based pages; replacement for sunset community loaders

Predict the output before running. The function definitions below match `examples/02_pdf_lazy.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Extract PDF pages lazily through a current core BaseLoader adapter.

Run: uv run --extra rag python modules/06_loaders_splitters/examples/02_pdf_lazy.py --offline
Expected: Ten pages with one-based page citations and handbook source.
Cost: No model calls or API fees.
Requires: rag
"""

from common.errors import run_cli
from common.knowledge import LocalLoader
from common.lesson import lesson_settings
from common.pretty import print_text


def main() -> None:
    lesson_settings()
    loader = LocalLoader("handbook/employee_handbook.pdf")
    # KEY IDEA: lazy_load yields documents; page numbers are normalized once at ingestion.
    for doc in loader.lazy_load():
        print_text(
            f"{doc.metadata['source']} page {doc.metadata['page']}: "
            f"{len(doc.page_content)} characters"
        )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Current BaseLoader adapters for CSV, text, Markdown and JSON; explicit directory ingestion

Predict the output before running. The function definitions below match `examples/03_file_loaders.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Load Markdown, CSV, JSON and an explicit policy directory.

Run: uv run --extra rag python modules/06_loaders_splitters/examples/03_file_loaders.py
--offline
Expected: Four policy files, 60 product rows, and 40 FAQ records.
Cost: No model calls or API fees.
Requires: rag
"""

from common.errors import run_cli
from common.knowledge import LocalLoader, policy_documents
from common.lesson import lesson_settings
from common.pretty import print_text


def main() -> None:
    lesson_settings()
    for filename in ("policies/returns.md", "products.csv", "faq.json"):
        docs = LocalLoader(filename).load()
        print_text(f"{filename}: {len(docs)} Documents; first metadata={docs[0].metadata}")
    # KEY IDEA: whitelist the policies directory; security fixtures are not ordinary evidence.
    print_text(f"Policy directory: {len(policy_documents(handbook=False))} Documents")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Web loading; local HTML fixture and explicit network mode

Predict the output before running. The function definitions below match `examples/04_web_loader.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Parse a local HTML fixture; fetch one allowlisted documentation URL only on opt-in.

Run: uv run --extra rag python modules/06_loaders_splitters/examples/04_web_loader.py --offline
Expected: Visible text from the local Brightwave HTML fixture; comments/scripts are excluded.
Cost: No model fee. --fetch-docs explicitly uses network; default is a local fixture.
Requires: rag
"""

import argparse

import httpx

from common.config import ConfigurationError, get_settings
from common.data import data_path
from common.errors import run_cli
from common.knowledge import html_document
from common.pretty import print_text


def main() -> None:
    # KEY IDEA: Fetching is explicit; visible-text extraction is not an injection defence.
    parser = argparse.ArgumentParser()
    parser.add_argument("--offline", action="store_true")
    parser.add_argument("--fetch-docs", action="store_true")
    args = parser.parse_args()
    source = "fixtures/policy.html"
    html = data_path(source).read_text(encoding="utf-8")
    if args.fetch_docs:
        if args.offline or get_settings().course_offline:
            raise ConfigurationError("--fetch-docs requires a live invocation without --offline.")
        source = "https://docs.langchain.com/oss/python/integrations/document_loaders"
        # Fixed URL, no arbitrary user-provided URL or redirects to private services.
        response = httpx.get(source, timeout=15, follow_redirects=False)
        response.raise_for_status()
        html = response.text[:200_000]
    print_text(html_document(html, source).page_content[:600])


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## RecursiveCharacterTextSplitter; chunk size/overlap visualization

Predict the output before running. The function definitions below match `examples/05_recursive_chunks.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Visualize recursive chunk boundaries and overlap on a policy.

Run: uv run --extra rag python modules/06_loaders_splitters/examples/05_recursive_chunks.py
--offline
Expected: Chunk sizes/start offsets and .runtime/chunk_overlap.png.
Cost: No model calls or API fees.
Requires: rag
"""

from common.config import ROOT
from common.errors import run_cli
from common.knowledge import LocalLoader, chunk_documents
from common.lesson import lesson_settings
from common.pretty import print_text


def main() -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    settings = lesson_settings()
    source = LocalLoader("policies/returns.md").load()
    chunks = chunk_documents(source, size=300, overlap=60)
    figure, axis = plt.subplots(figsize=(9, 4))
    for i, chunk in enumerate(chunks):
        start = chunk.metadata["start_index"]
        axis.barh(i, len(chunk.page_content), left=start, color="#167d8d")
        print_text(f"chunk {i}: start={start}, characters={len(chunk.page_content)}")
    # KEY IDEA: overlap preserves nearby context but duplicates indexed text and tokens.
    axis.set(
        xlabel="Character position in the policy",
        ylabel="Chunk",
        title="Recursive boundaries: target overlap 60 characters",
    )
    output = ROOT / settings.runtime_dir / "chunk_overlap.png"
    output.parent.mkdir(parents=True, exist_ok=True)
    figure.tight_layout()
    figure.savefig(output, dpi=150)
    plt.close(figure)
    print_text(str(output))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## MarkdownHeaderTextSplitter; heading and parent metadata

Predict the output before running. The function definitions below match `examples/06_markdown_chunks.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Preserve Markdown heading and parent metadata while limiting child size.

Run: uv run --extra rag python modules/06_loaders_splitters/examples/06_markdown_chunks.py
--offline
Expected: Chunks with heading, source, parent_id and chunk_id.
Cost: No model calls or API fees.
Requires: rag
"""

from common.errors import run_cli
from common.knowledge import LocalLoader, chunk_documents
from common.lesson import lesson_settings
from common.pretty import print_text


def main() -> None:
    lesson_settings()
    chunks = chunk_documents(
        LocalLoader("policies/returns.md").load(), strategy="markdown", size=350, overlap=50
    )
    # KEY IDEA: split headings first, then enforce size inside each logical section.
    for chunk in chunks:
        print_text(str(chunk.metadata))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Token-based splitting; code splitters; tokenizer limits

Predict the output before running. The function definitions below match `examples/07_token_code_chunks.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Compare a model tokenizer with an explicit offline token fixture and code-aware splitting.

Run: uv run --extra rag python modules/06_loaders_splitters/examples/07_token_code_chunks.py
--offline
Expected: Bounded token chunks and two small Python code sections.
Cost: No API fee; live tokenizer may download vocabulary once. Offline uses an explicit byte
fixture.
Requires: rag
"""

from langchain_text_splitters import Language, RecursiveCharacterTextSplitter, TokenTextSplitter
from langchain_text_splitters.base import Tokenizer, split_text_on_tokens

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


def main() -> None:
    # KEY IDEA: Token limits depend on the tokenizer; a byte fixture is not provider accounting.
    settings = lesson_settings()
    text = "Brightwave returns need proof of purchase. Refunds over $100 need approval."
    if settings.course_offline:
        # ASCII byte IDs make an inspectable test tokenizer, NOT a provider token count.
        fixture = Tokenizer(
            chunk_overlap=4,
            tokens_per_chunk=24,
            encode=lambda value: list(value.encode("ascii")),
            decode=lambda ids: bytes(ids).decode("ascii"),
        )
        chunks = split_text_on_tokens(text=text, tokenizer=fixture)
        print_text("OFFLINE byte-token fixture; not model token accounting.")
    else:
        # First use may download tiktoken's vocabulary; subsequent runs use its local cache.
        chunks = TokenTextSplitter(
            encoding_name="cl100k_base", chunk_size=24, chunk_overlap=4
        ).split_text(text)
    print_text(str(chunks))
    code = (
        "def eligible(days):\n    return days <= 30\n\n"
        "def approval(amount):\n    return amount > 100\n"
    )
    splitter = RecursiveCharacterTextSplitter.from_language(
        Language.PYTHON, chunk_size=60, chunk_overlap=0
    )
    print_text(str(splitter.split_text(code)))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world use case: Load policies and handbook with source/page metadata; compare three chunking strategies

Predict the output before running. The function definitions below match `use_case/knowledge_ingest.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Compare three chunking strategies over policy files and a ten-page handbook.

Run: uv run --extra rag python modules/06_loaders_splitters/use_case/knowledge_ingest.py
--offline
Expected: A JSON summary with chunk counts, sizes and provenance for each strategy.
Cost: No model calls or API fees.
Requires: rag
"""

import json

from common.config import ROOT
from common.errors import run_cli
from common.knowledge import chunk_documents, policy_documents
from common.lesson import lesson_settings
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()
    documents = policy_documents()
    report = {}
    for strategy in ("fixed", "recursive", "markdown"):
        chunks = chunk_documents(documents, strategy=strategy, size=450, overlap=80)
        report[strategy] = {
            "documents": len(documents),
            "chunks": len(chunks),
            "max_characters": max(len(d.page_content) for d in chunks),
            "indexed_characters": sum(len(d.page_content) for d in chunks),
            "sample_metadata": chunks[0].metadata,
        }
    output = ROOT / settings.runtime_dir / "ingest_comparison.json"
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print_text(json.dumps(report, indent=2))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run --extra rag pytest modules/06_loaders_splitters/tests` for the supplied solution checks.